# AEM4L5 E04 — Async para llamadas I/O

**Objetivo:** mostrar por qué varias esperas de red/storage no deberían ejecutarse siempre en serie.


## Secuencial vs async

```mermaid
sequenceDiagram
    participant App
    participant API1
    participant API2
    participant API3
    App->>API1: request 1
    API1-->>App: response 1
    App->>API2: request 2
    API2-->>App: response 2
    App->>API3: request 3
    API3-->>App: response 3
```

```mermaid
sequenceDiagram
    participant App
    participant API1
    participant API2
    participant API3
    App->>API1: request 1
    App->>API2: request 2
    App->>API3: request 3
    API1-->>App: response 1
    API2-->>App: response 2
    API3-->>App: response 3
```

Async ayuda cuando el programa está esperando I/O. No acelera cálculo CPU-bound puro.


In [ ]:
import asyncio
import time


def sync_fetch(doc_id: int) -> str:
    time.sleep(0.6)
    return f"doc_{doc_id}: resumen listo"


async def async_fetch(doc_id: int) -> str:
    await asyncio.sleep(0.6)
    return f"doc_{doc_id}: resumen listo"


start = time.perf_counter()
sync_results = [sync_fetch(i) for i in range(1, 6)]
sync_seconds = time.perf_counter() - start

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(*[async_fetch(i) for i in range(1, 6)])
    return results, time.perf_counter() - start

async_results, async_seconds = await main()

print("Sync:", round(sync_seconds, 2), sync_results)
print("Async:", round(async_seconds, 2), async_results)
print("Speedup:", round(sync_seconds / async_seconds, 2), "x")


## Cierre docente

La forma de pensar es: si varias tareas pasan la mayor parte del tiempo esperando, async permite solaparlas. Si todas queman CPU, async solo organiza la espera, no crea más CPU.
